# AnimationStudio — Phase 4: Animation Bible & Motion System on Google Colab

This notebook verifies and regenerates the **Phase-4 animation bible**
(`PHASE4.md` / `PHASE4_STATUS.md`) described by `scripts/generate_phase4.py`.

Unlike Phases 1–3 this phase produces **standards, not images** — the whole
run is pure Python:

- **No GPU / ComfyUI / model download required** (runs on the free CPU runtime).
- **No Drive mount needed** — nothing here touches `catalog.db`.
- Output: a refreshed `PHASE4_REPORT.md` (doc↔code consistency, resolved
  motion briefs, sample-shot validation) plus a green test run of the
  animation suites.

## What gets verified

| Deliverable | Where |
| --- | --- |
| Style guide / philosophy | `Animation/STYLE_GUIDE.md` + `src/animation_bible/` |
| Motion cycles (21), walk/run variants, jumps, dances | `src/animation_bible/libraries.py` |
| Facial library (13 expressions × 5 intensities, blinks, mouths) | same |
| Gestures (23), interactions (21) | same |
| Camera language (12 shots), transitions (11) | same |
| Physics rules (16), cloth elements (6), pacing tiers | same |
| Motion prompt templates + negative prompt | `src/animation_bible/prompts.py` |
| Quality checklist gate | `src/animation_bible/motion_system.py` |

## Steps

1. In Cell 1 set `REPO_URL` to your GitHub clone URL.
2. Runtime -> Run all.


In [ ]:
#@title 1. Settings

import os
import subprocess
import sys

# GitHub clone URL for this studio (push master there first).
REPO_URL = "https://github.com/YOUR_ORG/AnimationStudio.git"  #@param {type:"string"}
BRANCH = "master"  #@param ["master", "colab-gpu"]

WORK = "/content"
REPO = f"{WORK}/AnimationStudio"

# Cell 7: push the refreshed report back to GitHub. Off -> download a zip.
SYNC_TO_GITHUB = True  #@param {type:"boolean"}
GIT_NAME = "Colab Studio"  #@param {type:"string"}
GIT_EMAIL = "colab@animationstudio.local"  #@param {type:"string"}

# GitHub PAT (Settings -> Developer settings -> Tokens). Needs Contents:
# Read+Write. Leave empty if the repo is public and you push over HTTPS creds.
GITHUB_TOKEN = ""  #@param {type:"string"}


In [ ]:
#@title 2. Clone repo and install the studio

def run(cmd, **kw):
    print("+ " + " ".join(cmd))
    return subprocess.run(cmd, check=True, **kw)


os.chdir(WORK)
if not os.path.isdir(REPO):
    # Full clone so Cell 7 can push the refreshed report back.
    run(["git", "clone", "--branch", BRANCH, REPO_URL, "AnimationStudio"])
os.chdir(REPO)
run(["git", "checkout", BRANCH])
run(["git", "pull", "origin", BRANCH])

run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
run([sys.executable, "-m", "pip", "install", "-q",
     "fastapi", "uvicorn", "jinja2", "aiosqlite", "python-multipart",
     "pydantic", "scikit-learn"])
print("Studio installed (branch:", BRANCH, ")")
print("No GPU needed for Phase 4 — CPU runtime is fine.")


In [ ]:
#@title 3. Preview the bible scope (no generation)

from src.animation_bible import libraries as lib

rows = [
    ("Motion cycles", len(lib.MOTION_CYCLES)),
    ("Walk variants", len(lib.WALK_VARIANTS)),
    ("Run variants", len(lib.RUN_VARIANTS)),
    ("Jump cycles", len(lib.JUMP_CYCLES)),
    ("Dance loops", len(lib.DANCE_LOOPS)),
    ("Facial expressions", len(lib.FACIAL_EXPRESSIONS)),
    ("Blink types", len(lib.BLINK_TYPES)),
    ("Mouth shapes", len(lib.MOUTH_SHAPES)),
    ("Gestures", len(lib.GESTURES)),
    ("Interactions", len(lib.INTERACTIONS)),
    ("Camera shots", len(lib.CAMERA_SHOTS)),
    ("Scene transitions", len(lib.SCENE_TRANSITIONS)),
    ("Physics rules", len(lib.PHYSICS_RULES)),
    ("Cloth elements", len(lib.CLOTH_ELEMENTS)),
    ("Pacing standards", len(lib.PACING_STANDARDS)),
]
width = max(len(n) for n, _ in rows)
for name, count in rows:
    print(f"{name:<{width}} : {count}")
print(f"Master frame rate : {lib.MASTER_FRAME_RATE} fps "
      f"(export {lib.EXPORT_FRAME_RATE} fps)")


In [ ]:
#@title 4. Regenerate PHASE4_REPORT.md (doc↔code consistency + validation)

# Pure Python — verifies the Animation/ markdown bibles against the encoded
# library, resolves every motion into a MotionBrief, validates sample shots
# through the full motion system, and writes PHASE4_REPORT.md.
run([sys.executable, "scripts/generate_phase4.py"])


In [ ]:
#@title 5. Run the Phase-4 test suites

# tests/test_animation_bible.py — bible libraries, briefs, prompts,
#                                 motion system, doc↔code consistency.
# tests/test_animation.py       — executable motion pipeline engines.
run([sys.executable, "-m", "pytest",
     "tests/test_animation_bible.py", "tests/test_animation.py", "-q"])


In [ ]:
#@title 6. Review the report

from IPython.display import Markdown, display

with open(f"{REPO}/PHASE4_REPORT.md") as fh:
    report = fh.read()
display(Markdown(report[:6000]))
print("... (full file:", len(report), "chars)")


In [ ]:
#@title 7. Sync the refreshed report (GitHub push or manual download)

from datetime import datetime

if SYNC_TO_GITHUB:
    sys.path.insert(0, f"{REPO}/colab")
    from git_sync import _basic_auth_header

    def _run(cmd, **kw):
        print("+ " + " ".join(cmd))
        return subprocess.run(cmd, check=False, cwd=REPO, **kw)

    _run(["git", "config", "user.name", GIT_NAME])
    _run(["git", "config", "user.email", GIT_EMAIL])
    _run(["git", "add", "PHASE4_REPORT.md"])
    dirty = _run(["git", "status", "--porcelain"], capture_output=True, text=True)
    if dirty.stdout.strip():
        _run(["git", "commit", "-m",
              f"Phase 4 report {datetime.now():%Y-%m-%d %H:%M}"])
        if GITHUB_TOKEN:
            _run(["git", "-c",
                  f"http.extraheader=Authorization: {_basic_auth_header(GITHUB_TOKEN)}",
                  "push", "origin", BRANCH])
        else:
            _run(["git", "push", "origin", BRANCH])
    else:
        print("Report unchanged — nothing to push.")
else:
    from google.colab import files
    files.download(f"{REPO}/PHASE4_REPORT.md")
    print("Downloaded PHASE4_REPORT.md.")


## Next steps

- The refreshed `PHASE4_REPORT.md` now lives in your repo (or was downloaded).
- Browse the bible in the Review UI: **Dashboard → Motion** — every library
  above is rendered there, plus the text-only **Animation Prompt Builder**
  (`POST /motion/prompt`) for composing shot prompts without generating images.
- Image generation itself belongs to Phases 1–3 (`catalog.db` state) and the
  Phase 9 animation pipeline; this phase only constrains them.
- Re-run Cells 4–7 any time `Animation/*.md` or `src/animation_bible/` change.
